# Python reference 5: functions, decorators and typing

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/learning/python-reference-05-functions-and-typing.ipynb)

Parameter kinds, defaults, scope and closures; `functools` (`partial`, `reduce`, caching, `singledispatch`); decorators with and without arguments; `:=` and `match`; and type hints from basics to generics, `TypedDict` and `Protocol`. Practise it in [Python practice 5: functions, decorators and typing](../practice/python-practice-05-functions-and-typing.ipynb).

*Last reviewed: November 2026, against Python 3.13.*

## Contents

| section | entries |
| --- | --- |
| [Defining and calling functions](#defining-and-calling-functions) | [Parameter kinds](#parameter-kinds), [Default values are evaluated once](#default-values-are-evaluated-once), [Scope, closures and `nonlocal`](#scope-closures-and-nonlocal) |
| [Functional tools](#functional-tools) | [`lambda`, `key=` functions and `operator`](#lambda-key-functions-and-operator), [`functools`: `partial`, `reduce`, `cache`, `lru_cache`](#functools-partial-reduce-cache-lru_cache), [`functools.singledispatch`](#functoolssingledispatch) |
| [Decorators](#decorators) | [Writing a decorator](#writing-a-decorator), [Decorators with arguments](#decorators-with-arguments) |
| [Control flow extras](#control-flow-extras) | [Assignment expressions (`:=`)](#assignment-expressions-), [Structural pattern matching (`match`)](#structural-pattern-matching-match) |
| [Type hints](#type-hints) | [Annotating functions and variables](#annotating-functions-and-variables), [Generics, `TypedDict` and `Protocol`](#generics-typeddict-and-protocol) |

## Defining and calling functions

### Parameter kinds

```python
def f(pos_only, /, normal, *args, kw_only, kw_default=1, **kwargs): ...
```

| kind | syntax | how callers pass it |
| --- | --- | --- |
| positional-only | before `/` | by position only; the name is free to change |
| positional-or-keyword | plain name | either way |
| var-positional | `*args` | extra positional arguments, as a tuple |
| keyword-only | after `*` or `*args` | by name only |
| var-keyword | `**kwargs` | extra keyword arguments, as a dict |

**Notes:** keyword-only parameters make call sites self-describing (`load(path, overwrite=True)`) and let you add
options later without breaking positional callers. Positional-only parameters keep the parameter name out of your
public API (most builtins, like `len(obj, /)`, use them).

In [ ]:
def load(path, /, table, *, overwrite=False, **options):
    return f"{path} -> {table} overwrite={overwrite} options={options}"

print(load("s3://raw/orders", "orders", overwrite=True, compression="zstd"))
try:
    load(path="x", table="t")
except TypeError as e:
    print("TypeError:", e)
try:
    load("x", "t", True)
except TypeError as e:
    print("TypeError:", e)

def total(*amounts, rounding=2):
    return round(sum(amounts), rounding)

args, kwargs = [1.25, 2.5], {"rounding": 1}
print(total(*args), total(*args, **kwargs))

### Default values are evaluated once

A default is evaluated **when the `def` runs**, not on each call. A mutable default (`[]`, `{}`, `set()`) is therefore
shared by every call that does not pass the argument. Use `None` as the sentinel and create the object inside.
When `None` is itself a meaningful value, use a private sentinel object: `_MISSING = object()`.

In [ ]:
def append_bad(item, bucket=[]):
    bucket.append(item)
    return bucket

def append_good(item, bucket=None):
    bucket = [] if bucket is None else bucket
    bucket.append(item)
    return bucket

print(append_bad(1), append_bad(2), "| shared default:", append_bad.__defaults__)
print(append_good(1), append_good(2))

_MISSING = object()
def get(d, key, default=_MISSING):
    if key in d:
        return d[key]
    if default is _MISSING:
        raise KeyError(key)
    return default
print(get({"a": None}, "a"), get({}, "b", None))

### Scope, closures and `nonlocal`

Names resolve **LEGB**: local, enclosing function, global (module), builtins. Assigning to a name anywhere in a
function makes it local for the whole function unless declared `nonlocal` (enclosing function) or `global` (module).

A **closure** is an inner function that keeps the enclosing variables alive. Closures capture **variables, not
values**: functions created in a loop all see the loop variable's final value unless you bind it, for example as a
default argument (`lambda x, i=i: ...`) or with `functools.partial`.

In [ ]:
def make_counter():
    count = 0
    def increment(step=1):
        nonlocal count
        count += step
        return count
    return increment

counter = make_counter()
print(counter(), counter(), counter(10))

late = [lambda: i for i in range(3)]
bound = [lambda i=i: i for i in range(3)]
print([f() for f in late], [f() for f in bound])

rate = 0.2
def broken_tax(amount):
    try:
        rate = rate + 0.0   # assignment makes `rate` local, so the read fails
    except UnboundLocalError as e:
        return f"UnboundLocalError: {e}"
print(broken_tax(100))

## Functional tools

### `lambda`, `key=` functions and `operator`

`lambda args: expression` creates a small anonymous function. Its main use is as a `key=` for `sorted`, `min`, `max`,
`heapq.nlargest` and `itertools.groupby`. `operator.itemgetter`, `attrgetter` and `methodcaller` are faster, picklable
equivalents for common keys; `itemgetter("a", "b")` returns a tuple, which gives multi-field sorting.

In [ ]:
from operator import attrgetter, itemgetter, methodcaller

rows = [{"region": "south", "amount": 70}, {"region": "north", "amount": 120}, {"region": "north", "amount": 50}]
print(sorted(rows, key=itemgetter("region", "amount")))
print(max(rows, key=lambda r: r["amount"])["amount"], list(map(methodcaller("upper"), ["a", "b"])))
from datetime import date
print(attrgetter("year", "month")(date(2026, 11, 27)))

### `functools`: `partial`, `reduce`, `cache`, `lru_cache`

| tool | what it does |
| --- | --- |
| `partial(func, *args, **kwargs)` | a new callable with some arguments pre-filled |
| `reduce(func, iterable[, initial])` | folds left: `func(func(a, b), c)`; pass `initial` for empty input |
| `cache` (3.9+) | unbounded memoisation of a function on its (hashable) arguments |
| `lru_cache(maxsize=128)` | bounded memoisation; `f.cache_info()`, `f.cache_clear()` |
| `cached_property` | compute an attribute once per instance |

**Notes:** cached arguments must be hashable (convert lists to tuples). A cache on a method keeps `self` alive; prefer
`cached_property` or a module-level function. Caches hide changes in the underlying data, so clear them when it
changes.

In [ ]:
from functools import cache, lru_cache, partial, reduce

to_int_base2 = partial(int, base=2)
print(to_int_base2("1011"), reduce(lambda acc, x: acc * x, [1, 2, 3, 4]), reduce(lambda a, b: a + b, [], 0))

@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(80), fib.cache_info().hits)

@lru_cache(maxsize=2)
def lookup(key):
    print("  fetching", key)
    return key.upper()
lookup("a"); lookup("a"); lookup("b"); lookup("c"); lookup("a")
print(lookup.cache_info())

### `functools.singledispatch`

`@singledispatch` turns a function into a generic function that picks an implementation by the **type of the first
argument**. Register implementations with `@f.register` and a type annotation. It replaces long `isinstance` chains,
for example when serialising values of many types.

In [ ]:
from datetime import datetime
from decimal import Decimal
from functools import singledispatch

@singledispatch
def to_json_value(value):
    return value

@to_json_value.register
def _(value: Decimal):
    return str(value)

@to_json_value.register
def _(value: datetime):
    return value.isoformat()

@to_json_value.register
def _(value: set):
    return sorted(value)

print([to_json_value(v) for v in [1, Decimal("1.10"), datetime(2026, 11, 27, 9, 30), {"b", "a"}]])

## Decorators

### Writing a decorator

A decorator is a callable that takes a function and returns a replacement; `@decorator` above `def f` means
`f = decorator(f)`. Always wrap the inner function with `functools.wraps(func)` so the replacement keeps the original
`__name__`, `__doc__` and `__wrapped__` (logging, debugging and tests rely on them).

```python
def decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        ...  # before
        result = func(*args, **kwargs)
        ...  # after
        return result
    return wrapper
```

In [ ]:
import functools
import time

def timed(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            wrapper.last_seconds = time.perf_counter() - start
    return wrapper

@timed
def transform(rows):
    """Double every row."""
    return [r * 2 for r in rows]

print(transform([1, 2]), transform.__name__, transform.__doc__, transform.last_seconds < 1)

### Decorators with arguments

A decorator that takes options is a **factory**: a function that returns the decorator. `@retry(times=3)` calls
`retry(times=3)` first, and the result decorates the function. This is the shape of retry, rate-limit and
"register this handler" decorators.

In [ ]:
def retry(times=3, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions:
                    if attempt == times:
                        raise
        return wrapper
    return decorator

calls = {"n": 0}
@retry(times=3, exceptions=(ConnectionError,))
def flaky():
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("try again")
    return "ok"
print(flaky(), "after", calls["n"], "calls")

HANDLERS = {}
def handles(event):
    def register(func):
        HANDLERS[event] = func
        return func
    return register

@handles("order_placed")
def on_order(payload):
    return f"order {payload['id']}"
print(HANDLERS["order_placed"]({"id": 7}))

## Control flow extras

### Assignment expressions (`:=`)

`name := expression` assigns and returns the value inside an expression. Use it to avoid computing something twice
in a condition or comprehension, or in read-until-empty loops.

In [ ]:
import io, re

stream = io.StringIO("a\nb\n")
while (line := stream.readline()):
    print("line:", line.strip())
raw = ["id=7", "junk", "id=12"]
print([int(m.group(1)) for s in raw if (m := re.match(r"id=(\d+)", s))])

### Structural pattern matching (`match`)

`match subject:` tries each `case` pattern in order. Patterns can be literals, captures, wildcards (`_`), sequences
(`[first, *rest]`), mappings (`{"type": "order", **rest}`, extra keys allowed), class patterns
(`Point(x=0, y=y)`), alternatives (`"a" | "b"`) and guards (`case x if x > 0`).

**Notes:** a bare name in a pattern **captures**; it does not compare. Compare against constants with dotted names
(`Status.PAID`) or literals. Mapping patterns ignore extra keys; sequence patterns do not match strings.

In [ ]:
def describe(event):
    match event:
        case {"type": "order", "amount": amount} if amount >= 1000:
            return f"large order {amount}"
        case {"type": "order", "amount": amount}:
            return f"order {amount}"
        case {"type": "refund" | "chargeback" as kind}:
            return f"money back ({kind})"
        case [first, *rest]:
            return f"batch of {1 + len(rest)} starting {first}"
        case str() as text:
            return f"text {text!r}"
        case _:
            return "unknown"

for e in [{"type": "order", "amount": 1180, "id": 9}, {"type": "order", "amount": 30}, {"type": "chargeback"}, [1, 2, 3], "ping", 42]:
    print(describe(e))

## Type hints

### Annotating functions and variables

Annotations document intent and let tools (mypy, pyright, IDEs) check code. Python itself **does not enforce them**
at runtime.

| hint | meaning |
| --- | --- |
| `int`, `str`, `list[int]`, `dict[str, float]`, `tuple[int, ...]` | builtins as generics (3.9+) |
| `X \| None` | optional (3.10+; older code writes `Optional[X]`) |
| `A \| B` | union |
| `Literal["append", "overwrite"]` | one of these exact values |
| `Callable[[int, str], bool]` | a callable with that signature |
| `Iterable[T]`, `Iterator[T]`, `Mapping[K, V]`, `Sequence[T]` | abstract types from `collections.abc`; accept these, return concrete ones |
| `Any` | opt out of checking |
| `TypeAlias` / `type Name = ...` (3.12+) | name a complex type |

**Notes:** accept the loosest type you need (`Iterable[str]`) and return the most specific (`list[str]`).
`typing.get_type_hints(f)` resolves annotations at runtime, which libraries such as pydantic use.

In [ ]:
from collections.abc import Callable, Iterable
from typing import Literal, get_type_hints

Mode = Literal["append", "overwrite"]

def write(rows: Iterable[dict[str, object]], mode: Mode = "append", on_error: Callable[[Exception], None] | None = None) -> int:
    return sum(1 for _ in rows)

print(write([{"a": 1}], "overwrite"), get_type_hints(write)["mode"])
print(write([{"a": 1}], "not-a-mode"), "<- runs anyway: hints are not enforced")

### Generics, `TypedDict` and `Protocol`

| tool | use |
| --- | --- |
| `def first[T](xs: list[T]) -> T` (3.12+) or `T = TypeVar("T")` | a function that works for any element type and keeps it |
| `class Box[T]:` (3.12+) or `Generic[T]` | a generic class |
| `TypedDict` | the shape of a dict with known keys (JSON records); `NotRequired[...]` for optional keys |
| `Protocol` | structural typing: anything with these methods matches, no inheritance needed |

**Notes:** `TypedDict` values are ordinary dicts at runtime; there is no validation (use pydantic or a dataclass with
checks for that). Classes, dataclasses and protocols are covered in
[classes and dataclasses](python-reference-07-classes-and-dataclasses.ipynb).

In [ ]:
from typing import NotRequired, Protocol, TypedDict

def first[T](items: list[T], default: T) -> T:
    return items[0] if items else default

class Order(TypedDict):
    order_id: int
    status: str
    coupon: NotRequired[str]

class SupportsRead(Protocol):
    def read(self, size: int = -1) -> str: ...

def head(source: SupportsRead, n: int) -> str:
    return source.read(n)

order: Order = {"order_id": 7, "status": "paid"}
print(first([3, 4], 0), first([], "none"), order, type(order).__name__, head(io.StringIO("hello world"), 5))

**Next:** practise these entries in [Python practice 5: functions, decorators and typing](../practice/python-practice-05-functions-and-typing.ipynb).